# Notebook 07: Mining Economics

## Overview

This notebook provides a rigorous, quantitative exploration of **cryptocurrency mining economics**. We model mining profitability from first principles, simulate Bitcoin's difficulty adjustment algorithm, analyze the block reward and supply schedule, compare mining pool payout schemes, and evaluate the cost of executing a 51% attack.

All analysis is performed with pure computation -- no external APIs or network connections are required.

## Prerequisites

- **Notebook 01**: Proof-of-Work basics (hash functions, nonce searching, difficulty targets)
- Familiarity with basic probability and statistics
- Python proficiency (NumPy, Pandas, Matplotlib)

## Learning Objectives

By the end of this notebook you will be able to:

1. **Calculate** mining profitability given hardware specs, electricity costs, and network parameters
2. **Simulate** Bitcoin's difficulty adjustment mechanism and predict its response to hash rate changes
3. **Model** the complete Bitcoin supply schedule including halvings and inflation rate
4. **Compare** mining pool payout schemes (PPS vs. PPLNS) and quantify their variance characteristics
5. **Estimate** the economic cost of a 51% attack and understand how it scales with network size

## Estimated Time: 4-6 hours

---

In [ ]:
# === Setup ===
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Reproducibility
np.random.seed(42)

print('All libraries loaded successfully.')

---
# Part 1: Mining Profitability Calculator

Mining profitability depends on the interplay between **revenue** (block rewards + fees) and **costs** (hardware, electricity, cooling, overhead). We build a calculator from first principles.

### Key Formula

The expected number of blocks a miner finds per day is:

$$\text{Blocks/day} = \frac{\text{Miner Hash Rate}}{\text{Network Hash Rate}} \times 144$$

where 144 = expected blocks per day (one every ~10 minutes). Equivalently, using difficulty $D$:

$$\text{Revenue/day} = \frac{H \times 86400}{D \times 2^{32}} \times R$$

where $H$ is hash rate (H/s), $D$ is difficulty, and $R$ is block reward in BTC.

In [ ]:
# === Mining Profitability Calculator ===

def mining_profitability(
    hash_rate_th: float,        # Miner hash rate in TH/s
    power_watts: float,         # Power consumption in Watts
    electricity_cost: float,    # USD per kWh
    block_reward: float,        # BTC per block
    difficulty: float,          # Current network difficulty
    btc_price: float,           # USD per BTC
    pool_fee: float = 0.02,     # Pool fee (default 2%)
    avg_tx_fees: float = 0.5    # Average transaction fees per block in BTC
) -> dict:
    """
    Calculate mining profitability over daily, monthly, and yearly periods.
    """
    hash_rate_hs = hash_rate_th * 1e12  # Convert TH/s to H/s
    
    # Expected BTC mined per day
    # blocks_per_day = (hash_rate * seconds_per_day) / (difficulty * 2^32)
    blocks_per_day = (hash_rate_hs * 86400) / (difficulty * 2**32)
    btc_per_day = blocks_per_day * (block_reward + avg_tx_fees)
    
    # Revenue (after pool fee)
    daily_revenue_btc = btc_per_day * (1 - pool_fee)
    daily_revenue_usd = daily_revenue_btc * btc_price
    
    # Electricity cost
    daily_kwh = (power_watts * 24) / 1000
    daily_elec_cost = daily_kwh * electricity_cost
    
    # Profit
    daily_profit = daily_revenue_usd - daily_elec_cost
    
    return {
        'btc_per_day': daily_revenue_btc,
        'revenue_daily': daily_revenue_usd,
        'revenue_monthly': daily_revenue_usd * 30,
        'revenue_yearly': daily_revenue_usd * 365,
        'elec_cost_daily': daily_elec_cost,
        'elec_cost_monthly': daily_elec_cost * 30,
        'elec_cost_yearly': daily_elec_cost * 365,
        'profit_daily': daily_profit,
        'profit_monthly': daily_profit * 30,
        'profit_yearly': daily_profit * 365,
        'daily_kwh': daily_kwh,
        'blocks_per_day': blocks_per_day
    }


# --- Real ASIC Specs: Bitmain Antminer S21 ---
asic_specs = {
    'name': 'Antminer S21',
    'hash_rate_th': 200,      # 200 TH/s
    'power_watts': 3500,      # 3500 W
    'efficiency': 3500 / 200, # J/TH
    'price_usd': 5000         # Approximate unit cost
}

# Representative network parameters
network_params = {
    'difficulty': 75e12,        # ~75 trillion (representative)
    'block_reward': 3.125,      # Post-2024 halving
    'btc_price': 60000,         # USD
    'electricity_cost': 0.07,   # USD/kWh (competitive industrial rate)
}

result = mining_profitability(
    hash_rate_th=asic_specs['hash_rate_th'],
    power_watts=asic_specs['power_watts'],
    electricity_cost=network_params['electricity_cost'],
    block_reward=network_params['block_reward'],
    difficulty=network_params['difficulty'],
    btc_price=network_params['btc_price']
)

print(f"=== {asic_specs['name']} Mining Profitability ===")
print(f"Hash Rate: {asic_specs['hash_rate_th']} TH/s | Power: {asic_specs['power_watts']}W")
print(f"Efficiency: {asic_specs['efficiency']:.1f} J/TH")
print(f"\nBTC Price: ${network_params['btc_price']:,.0f} | Difficulty: {network_params['difficulty']:.2e}")
print(f"Electricity: ${network_params['electricity_cost']}/kWh\n")
print(f"{'Period':<12} {'Revenue':>12} {'Elec Cost':>12} {'Profit':>12}")
print('-' * 50)
for period in ['daily', 'monthly', 'yearly']:
    rev = result[f'revenue_{period}']
    cost = result[f'elec_cost_{period}']
    profit = result[f'profit_{period}']
    print(f"{period.capitalize():<12} ${rev:>11,.2f} ${cost:>11,.2f} ${profit:>11,.2f}")

print(f"\nBTC mined per day: {result['btc_per_day']:.6f}")
print(f"Expected blocks/day: {result['blocks_per_day']:.6f}")
print(f"Daily electricity: {result['daily_kwh']:.1f} kWh")

### Sensitivity Analysis

Mining profitability is highly sensitive to two key variables: **BTC price** and **electricity cost**. Let's create a heatmap to visualize the profit landscape.

In [ ]:
# === Sensitivity Analysis: Profit vs BTC Price and Electricity Cost ===

btc_prices = np.arange(20000, 120001, 5000)
elec_costs = np.arange(0.02, 0.16, 0.01)

profit_matrix = np.zeros((len(elec_costs), len(btc_prices)))

for i, elec in enumerate(elec_costs):
    for j, price in enumerate(btc_prices):
        r = mining_profitability(
            hash_rate_th=200, power_watts=3500,
            electricity_cost=elec, block_reward=3.125,
            difficulty=75e12, btc_price=price
        )
        profit_matrix[i, j] = r['profit_daily']

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(profit_matrix, aspect='auto', cmap='RdYlGn', origin='lower',
               extent=[btc_prices[0], btc_prices[-1], elec_costs[0], elec_costs[-1]])

# Add break-even contour
cs = ax.contour(btc_prices, elec_costs, profit_matrix, levels=[0],
                colors='black', linewidths=2, linestyles='dashed')
ax.clabel(cs, fmt='Break-even', fontsize=10)

ax.set_xlabel('BTC Price (USD)', fontsize=13)
ax.set_ylabel('Electricity Cost (USD/kWh)', fontsize=13)
ax.set_title('Antminer S21 Daily Profit (USD) -- Sensitivity Analysis', fontsize=14)
plt.colorbar(im, ax=ax, label='Daily Profit (USD)')
plt.tight_layout()
plt.show()

print("The dashed black line shows the break-even boundary.")
print("Above the line: unprofitable. Below the line: profitable.")

In [ ]:
# === Break-Even Analysis ===

# Break-even BTC price for different electricity costs
elec_range = np.linspace(0.02, 0.15, 100)
breakeven_prices = []

for elec in elec_range:
    daily_elec = (3500 * 24 / 1000) * elec
    # Revenue = (H * 86400) / (D * 2^32) * (reward + fees) * (1 - pool_fee) * btc_price
    btc_per_day = (200e12 * 86400) / (75e12 * 2**32) * 3.625 * 0.98
    breakeven_price = daily_elec / btc_per_day
    breakeven_prices.append(breakeven_price)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(elec_range * 100, breakeven_prices, 'b-', linewidth=2.5)
ax.fill_between(elec_range * 100, breakeven_prices, max(breakeven_prices) * 1.1,
                alpha=0.15, color='red', label='Unprofitable zone')
ax.fill_between(elec_range * 100, 0, breakeven_prices,
                alpha=0.15, color='green', label='Profitable zone')

# Mark common electricity rates
markers = [(0.04, 'Kazakhstan'), (0.07, 'Texas'), (0.10, 'US Avg'), (0.13, 'Germany')]
for rate, label in markers:
    idx = np.argmin(np.abs(elec_range - rate))
    ax.plot(rate * 100, breakeven_prices[idx], 'ko', markersize=8)
    ax.annotate(f'{label}\n${rate:.2f}/kWh', (rate * 100, breakeven_prices[idx]),
                textcoords='offset points', xytext=(10, 10), fontsize=9)

ax.set_xlabel('Electricity Cost (cents/kWh)', fontsize=13)
ax.set_ylabel('Break-Even BTC Price (USD)', fontsize=13)
ax.set_title('Break-Even BTC Price vs Electricity Cost (Antminer S21)', fontsize=14)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

**Key Insight**: Mining profitability is a razor-thin margin business. A miner in Kazakhstan ($0.04/kWh) can remain profitable at much lower BTC prices than a miner in Germany ($0.13/kWh). This explains the geographic concentration of mining operations in regions with cheap electricity.

---
# Part 2: Difficulty Adjustment Simulation

Bitcoin adjusts its mining difficulty every **2,016 blocks** (~2 weeks) to maintain a target block time of 10 minutes. The adjustment formula is:

$$D_{\text{new}} = D_{\text{old}} \times \frac{\text{Actual Time for 2016 Blocks}}{\text{Target Time (2 weeks)}}$$

With a cap: the difficulty can change by at most a factor of 4 in either direction per adjustment.

In [ ]:
# === Difficulty Adjustment Algorithm ===

TARGET_BLOCK_TIME = 600  # 10 minutes in seconds
BLOCKS_PER_PERIOD = 2016
TARGET_PERIOD_TIME = TARGET_BLOCK_TIME * BLOCKS_PER_PERIOD  # 2 weeks in seconds
MAX_ADJUSTMENT_FACTOR = 4.0  # Maximum 4x change per period


def adjust_difficulty(current_difficulty: float, actual_time: float) -> float:
    """
    Bitcoin difficulty adjustment algorithm.
    
    Parameters:
        current_difficulty: Current difficulty value
        actual_time: Actual time (seconds) taken to mine 2016 blocks
    Returns:
        New difficulty value
    """
    ratio = actual_time / TARGET_PERIOD_TIME
    # Clamp to [1/4, 4] range
    ratio = max(1.0 / MAX_ADJUSTMENT_FACTOR, min(MAX_ADJUSTMENT_FACTOR, ratio))
    return current_difficulty * ratio


def expected_block_time(difficulty: float, network_hash_rate: float) -> float:
    """
    Expected seconds to find one block given difficulty and hash rate.
    E[T] = (D * 2^32) / H
    """
    return (difficulty * 2**32) / network_hash_rate


# Quick test
d = 75e12
h = 500e18  # 500 EH/s
t = expected_block_time(d, h)
print(f"Difficulty: {d:.2e}")
print(f"Network hash rate: {h/1e18:.0f} EH/s")
print(f"Expected block time: {t:.1f} seconds ({t/60:.1f} minutes)")

In [ ]:
# === Simulate 60 Difficulty Periods with Varying Hash Rate ===

def simulate_difficulty_periods(n_periods: int, initial_difficulty: float,
                                 hash_rate_schedule: callable) -> pd.DataFrame:
    """
    Simulate n difficulty adjustment periods.
    
    hash_rate_schedule: function(period_index) -> hash_rate in H/s
    """
    records = []
    difficulty = initial_difficulty
    cumulative_blocks = 0
    cumulative_time = 0  # seconds
    
    for period in range(n_periods):
        hash_rate = hash_rate_schedule(period)
        
        # Expected block time with current difficulty and hash rate
        block_time = expected_block_time(difficulty, hash_rate)
        
        # Time to mine 2016 blocks
        actual_time = block_time * BLOCKS_PER_PERIOD
        
        records.append({
            'period': period,
            'difficulty': difficulty,
            'hash_rate_ehs': hash_rate / 1e18,
            'avg_block_time_min': block_time / 60,
            'period_duration_days': actual_time / 86400,
            'cumulative_blocks': cumulative_blocks,
            'cumulative_days': cumulative_time / 86400
        })
        
        cumulative_blocks += BLOCKS_PER_PERIOD
        cumulative_time += actual_time
        
        # Adjust difficulty
        difficulty = adjust_difficulty(difficulty, actual_time)
    
    return pd.DataFrame(records)


# Hash rate schedule: gradual growth with a sudden 50% drop at period 35
def hash_rate_schedule(period):
    base = 400e18  # 400 EH/s starting point
    if period < 35:
        # Gradual 3% growth per period
        return base * (1.03 ** period)
    elif period == 35:
        # Sudden 50% drop (e.g., China mining ban scenario)
        return base * (1.03 ** 34) * 0.5
    else:
        # Recovery: 5% growth from the dropped level
        dropped = base * (1.03 ** 34) * 0.5
        return dropped * (1.05 ** (period - 35))


df_diff = simulate_difficulty_periods(60, 75e12, hash_rate_schedule)
print(f"Simulated {len(df_diff)} difficulty periods")
print(f"Time span: {df_diff['cumulative_days'].iloc[-1]:.0f} days "
      f"({df_diff['cumulative_days'].iloc[-1]/365:.1f} years)")
df_diff.head()

In [ ]:
# === Plot: Difficulty vs Hash Rate Over Simulated Time ===

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

days = df_diff['cumulative_days']

# Hash rate
ax1.plot(days, df_diff['hash_rate_ehs'], 'b-o', markersize=3, linewidth=1.5)
ax1.axvline(x=days.iloc[35], color='red', linestyle='--', alpha=0.7, label='50% hash rate drop')
ax1.set_ylabel('Hash Rate (EH/s)', fontsize=12)
ax1.set_title('Difficulty Adjustment Simulation (60 Periods)', fontsize=14)
ax1.legend()

# Difficulty
ax2.plot(days, df_diff['difficulty'], 'g-o', markersize=3, linewidth=1.5)
ax2.axvline(x=days.iloc[35], color='red', linestyle='--', alpha=0.7)
ax2.set_ylabel('Difficulty', fontsize=12)
ax2.ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))

# Average block time
ax3.plot(days, df_diff['avg_block_time_min'], 'm-o', markersize=3, linewidth=1.5)
ax3.axhline(y=10, color='gray', linestyle=':', alpha=0.7, label='10-min target')
ax3.axvline(x=days.iloc[35], color='red', linestyle='--', alpha=0.7)
ax3.set_ylabel('Avg Block Time (min)', fontsize=12)
ax3.set_xlabel('Cumulative Days', fontsize=12)
ax3.legend()

plt.tight_layout()
plt.show()

# Impact analysis
pre_drop = df_diff.iloc[34]
post_drop = df_diff.iloc[35]
print(f"\n=== Impact of 50% Hash Rate Drop ===")
print(f"Hash rate: {pre_drop['hash_rate_ehs']:.1f} -> {post_drop['hash_rate_ehs']:.1f} EH/s")
print(f"Block time: {pre_drop['avg_block_time_min']:.1f} -> {post_drop['avg_block_time_min']:.1f} min")
print(f"Period duration: {pre_drop['period_duration_days']:.1f} -> {post_drop['period_duration_days']:.1f} days")
print(f"\nDifficulty adjusts down in the NEXT period to compensate.")

**Key Insight**: When 50% of the hash rate drops suddenly, block times **double** to ~20 minutes until the next difficulty adjustment. The adjustment algorithm then corrects, but it takes a full 2016-block period (~4 weeks at the slower rate) before blocks return to the 10-minute target. This lag is an important property of Bitcoin's self-regulating mechanism.

---
# Part 3: Block Reward and Supply Schedule

Bitcoin's monetary policy is defined entirely in code. The block subsidy starts at **50 BTC** and halves every **210,000 blocks** (~4 years). This creates a predictable, disinflationary supply schedule converging to 21 million BTC.

In [ ]:
# === Block Reward / Halving Schedule ===

INITIAL_REWARD = 50.0  # BTC
HALVING_INTERVAL = 210_000  # blocks
SATOSHIS_PER_BTC = 1e8


def block_reward(height: int) -> float:
    """Return the block subsidy (BTC) at a given block height."""
    n_halvings = height // HALVING_INTERVAL
    if n_halvings >= 64:  # After 64 halvings, reward is effectively 0
        return 0.0
    # Use integer arithmetic on satoshis to avoid float issues
    reward_sats = int(INITIAL_REWARD * SATOSHIS_PER_BTC) >> n_halvings
    return reward_sats / SATOSHIS_PER_BTC


def cumulative_supply(height: int) -> float:
    """Calculate total BTC supply at a given block height."""
    supply = 0.0
    remaining_blocks = height
    halving = 0
    while remaining_blocks > 0 and halving < 64:
        blocks_in_era = min(remaining_blocks, HALVING_INTERVAL)
        reward = block_reward(halving * HALVING_INTERVAL)
        supply += blocks_in_era * reward
        remaining_blocks -= blocks_in_era
        halving += 1
    return supply


# Display halving schedule
print(f"{'Halving':>8} {'Block Height':>14} {'Reward (BTC)':>14} {'~Year':>8} {'Supply (M BTC)':>16}")
print('-' * 64)
for i in range(10):
    h = i * HALVING_INTERVAL
    r = block_reward(h)
    s = cumulative_supply(h) / 1e6
    year = 2009 + i * 4  # Approximate
    if r == 0:
        break
    print(f"{i:>8} {h:>14,} {r:>14.8f} {year:>8} {s:>16.4f}")

print(f"\nTheoretical maximum supply: {cumulative_supply(100 * HALVING_INTERVAL):,.8f} BTC")

In [ ]:
# === Plot: Cumulative Supply Curve and Inflation Rate ===

# Generate data points
heights = np.arange(0, 10 * HALVING_INTERVAL, 1000)
years = 2009 + heights / (365.25 * 144)  # Approximate calendar year

supplies = np.array([cumulative_supply(int(h)) for h in tqdm(heights, desc='Computing supply')])

# Annual inflation rate
blocks_per_year = 365.25 * 144
annual_new_btc = np.array([block_reward(int(h)) * blocks_per_year for h in heights])
inflation_rate = np.where(supplies > 0, (annual_new_btc / supplies) * 100, 0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Supply curve
ax1.plot(years, supplies / 1e6, 'b-', linewidth=2)
ax1.axhline(y=21, color='red', linestyle='--', alpha=0.5, label='21M cap')
ax1.set_ylabel('Cumulative Supply (Millions BTC)', fontsize=12)
ax1.set_title('Bitcoin Supply Schedule', fontsize=14)
ax1.legend(fontsize=11)

# Mark halvings
for i in range(8):
    halving_year = 2009 + i * 4
    if halving_year <= years[-1]:
        ax1.axvline(x=halving_year, color='gray', linestyle=':', alpha=0.3)

# Inflation rate
ax2.plot(years, inflation_rate, 'orange', linewidth=2, label='Bitcoin')
ax2.axhline(y=1.5, color='goldenrod', linestyle='--', alpha=0.5, label='Gold (~1.5%/yr)')
ax2.set_ylabel('Annual Inflation Rate (%)', fontsize=12)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylim(0, 25)
ax2.legend(fontsize=11)
ax2.set_title('Bitcoin vs Gold Inflation Rate', fontsize=14)

plt.tight_layout()
plt.show()

# When does Bitcoin's inflation drop below gold's?
below_gold = years[inflation_rate < 1.5]
if len(below_gold) > 0:
    print(f"Bitcoin inflation drops below gold (~1.5%) around year {below_gold[0]:.0f}")

In [ ]:
# === Gold vs Bitcoin Production Comparison ===

# Gold: ~3,300 tonnes mined per year, ~205,000 tonnes above-ground total
# Simplified model: linear gold production (roughly constant for past decades)

gold_total_2009 = 165_000  # tonnes in 2009 (approximate)
gold_annual_production = 3_300  # tonnes per year

btc_years = np.arange(2009, 2045)
btc_supply_pct = np.array([cumulative_supply(int((y - 2009) * blocks_per_year)) / 21e6 * 100
                           for y in btc_years])

# Gold supply as % of estimated total (in-ground + above-ground ~ 244,000 tonnes)
gold_estimated_total = 244_000
gold_supply_pct = np.array([(gold_total_2009 + (y - 2009) * gold_annual_production) / gold_estimated_total * 100
                             for y in btc_years])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(btc_years, btc_supply_pct, 'b-', linewidth=2.5, label='Bitcoin (% of 21M)')
ax.plot(btc_years, gold_supply_pct, '--', color='goldenrod', linewidth=2.5,
        label='Gold (% of est. total reserves)')
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('% of Total Supply Extracted', fontsize=13)
ax.set_title('Bitcoin vs Gold: Extraction Progress', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

**Key Insight**: Bitcoin's supply schedule is algorithmically predetermined and fully transparent. By contrast, gold production depends on exploration, technology, and market prices. Bitcoin's inflation rate drops below gold's around the 3rd halving (~2020), making it increasingly "harder" money in stock-to-flow terms.

---
# Part 4: Mining Pool Economics

Solo mining has extremely high variance. A single Antminer S21 might wait **years** between finding blocks. Mining pools aggregate hash rate and share rewards. The two most common payout schemes are:

- **PPS (Pay Per Share)**: The pool pays a fixed amount for each valid share submitted, regardless of whether the pool finds a block. The pool absorbs variance.
- **PPLNS (Pay Per Last N Shares)**: When a block is found, rewards are distributed proportionally to shares submitted in the last N shares window. Miners absorb some variance.

In [ ]:
# === PPS Reward Calculation ===

def pps_reward_per_share(difficulty: float, block_reward_btc: float,
                          pool_fee: float = 0.02) -> float:
    """
    Calculate the PPS reward per share.
    Each share at pool difficulty D_share has probability 1/D_share of being the winning hash.
    PPS pay = block_reward / difficulty * (1 - fee)
    """
    return block_reward_btc / difficulty * (1 - pool_fee)


def pps_daily_income(miner_hashrate_th: float, share_difficulty: float,
                      network_difficulty: float, block_reward_btc: float,
                      pool_fee: float = 0.02) -> float:
    """
    Calculate daily PPS income for a miner.
    """
    # Shares per second = hash_rate / (share_difficulty * 2^32)
    hash_rate_hs = miner_hashrate_th * 1e12
    shares_per_second = hash_rate_hs / (share_difficulty * 2**32)
    shares_per_day = shares_per_second * 86400
    
    # PPS value of each share (based on network difficulty)
    value_per_share = block_reward_btc / (network_difficulty / share_difficulty) * (1 - pool_fee)
    
    return shares_per_day * value_per_share


# Example
share_diff = 1e6  # Pool share difficulty
daily_pps = pps_daily_income(200, share_diff, 75e12, 3.125)
print(f"PPS daily income (200 TH/s): {daily_pps:.8f} BTC")
print(f"PPS monthly income: {daily_pps * 30:.6f} BTC")

In [ ]:
# === PPLNS Simulation ===

def simulate_mining_pool(n_blocks: int, miners: dict, network_difficulty: float,
                          block_reward_btc: float, pplns_window: int = 10,
                          pool_fee: float = 0.02) -> dict:
    """
    Simulate mining pool operation over n_blocks.
    
    miners: dict of {name: hash_rate_th}
    Returns dict with PPS earnings, PPLNS earnings, and solo earnings per miner.
    """
    total_pool_hashrate = sum(miners.values())
    miner_names = list(miners.keys())
    miner_fractions = {name: hr / total_pool_hashrate for name, hr in miners.items()}
    
    # Pool's fraction of network
    # Assume network is 500 EH/s total
    network_hashrate_th = 500e6  # 500 EH/s in TH/s
    pool_fraction = total_pool_hashrate / network_hashrate_th
    
    # Initialize earnings
    solo_earnings = {name: 0.0 for name in miner_names}
    pps_earnings = {name: 0.0 for name in miner_names}
    pplns_earnings = {name: 0.0 for name in miner_names}
    
    # Track shares for PPLNS
    share_history = []  # list of (miner_name, block_number)
    pool_blocks_found = []
    
    for block in range(n_blocks):
        # Each miner submits shares proportional to their hash rate
        for name in miner_names:
            n_shares = max(1, int(miners[name] / 10))  # Simplified share count
            for _ in range(n_shares):
                share_history.append((name, block))
        
        # Did the pool find this block?
        if np.random.random() < pool_fraction:
            pool_blocks_found.append(block)
            reward_after_fee = block_reward_btc * (1 - pool_fee)
            
            # PPLNS: distribute based on last N blocks of shares
            window_start = max(0, block - pplns_window)
            recent_shares = [(name, b) for name, b in share_history if b >= window_start]
            share_counts = {name: 0 for name in miner_names}
            for name, _ in recent_shares:
                share_counts[name] += 1
            total_shares = sum(share_counts.values())
            
            if total_shares > 0:
                for name in miner_names:
                    pplns_earnings[name] += reward_after_fee * share_counts[name] / total_shares
        
        # Solo mining: each miner independently
        for name in miner_names:
            solo_prob = (miners[name] * 1e12) / (network_hashrate_th * 1e12)
            if np.random.random() < solo_prob:
                solo_earnings[name] += block_reward_btc
        
        # PPS: guaranteed pay per share (deterministic)
        for name in miner_names:
            expected_btc = (miners[name] / network_hashrate_th) * block_reward_btc * (1 - pool_fee)
            pps_earnings[name] += expected_btc
    
    return {
        'solo': solo_earnings,
        'pps': pps_earnings,
        'pplns': pplns_earnings,
        'pool_blocks': len(pool_blocks_found),
        'total_blocks': n_blocks
    }


# Define miners in the pool
miners = {
    'Large Farm (5000 TH/s)': 5000,
    'Medium Farm (1000 TH/s)': 1000,
    'Small Miner A (200 TH/s)': 200,
    'Small Miner B (200 TH/s)': 200,
    'Hobbyist (50 TH/s)': 50,
}

print("Running pool simulation (500 blocks)...")
result_pool = simulate_mining_pool(500, miners, 75e12, 3.125)
print(f"Pool found {result_pool['pool_blocks']} blocks out of {result_pool['total_blocks']}")

In [ ]:
# === Compare Variance: Solo vs PPS vs PPLNS ===

# Run multiple simulations to build distributions
n_simulations = 200
n_blocks_per_sim = 500

# Focus on the small miner for variance comparison
target_miner = 'Small Miner A (200 TH/s)'

solo_results = []
pps_results = []
pplns_results = []

for _ in tqdm(range(n_simulations), desc='Simulations'):
    r = simulate_mining_pool(n_blocks_per_sim, miners, 75e12, 3.125)
    solo_results.append(r['solo'][target_miner])
    pps_results.append(r['pps'][target_miner])
    pplns_results.append(r['pplns'][target_miner])

# Summary statistics
print(f"\n=== Earnings Distribution for {target_miner} ({n_blocks_per_sim} blocks) ===")
print(f"{'Method':<10} {'Mean (BTC)':>12} {'Std Dev':>12} {'CV (%)':>10} {'Min':>10} {'Max':>10}")
print('-' * 66)
for label, data in [('Solo', solo_results), ('PPS', pps_results), ('PPLNS', pplns_results)]:
    arr = np.array(data)
    cv = (arr.std() / arr.mean() * 100) if arr.mean() > 0 else float('inf')
    print(f"{label:<10} {arr.mean():>12.6f} {arr.std():>12.6f} {cv:>10.1f} {arr.min():>10.6f} {arr.max():>10.6f}")

In [ ]:
# === Histogram Comparison ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, label, color in zip(
    axes,
    [solo_results, pps_results, pplns_results],
    ['Solo Mining', 'PPS (Pay Per Share)', 'PPLNS'],
    ['#e74c3c', '#2ecc71', '#3498db']
):
    arr = np.array(data)
    ax.hist(arr, bins=25, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(arr.mean(), color='black', linestyle='--', linewidth=2, label=f'Mean: {arr.mean():.6f}')
    ax.set_title(label, fontsize=13)
    ax.set_xlabel('Earnings (BTC)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.legend(fontsize=9)

fig.suptitle(f'Earnings Distribution: {target_miner} over {n_blocks_per_sim} Blocks',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Solo mining shows extreme variance (often zero earnings).")
print("PPS provides the most predictable income stream.")
print("PPLNS falls between the two, with moderate variance.")

**Key Insight**: For small miners, solo mining is essentially a lottery. PPS provides guaranteed income (the pool acts as an insurer and charges a premium via fees). PPLNS offers a middle ground with lower fees but higher variance. The coefficient of variation (CV) quantifies this tradeoff clearly.

---
# Part 5: 51% Attack Cost Analysis

A 51% attack occurs when an entity controls more than half the network's hash rate, enabling them to:
- **Double-spend** transactions
- **Censor** transactions
- **Reorganize** recent blocks

We analyze the economic cost of mounting such an attack.

In [ ]:
# === 51% Attack Cost Model ===

def attack_cost_analysis(
    network_hashrate_ehs: float,  # Network hash rate in EH/s
    asic_hashrate_th: float = 200,  # Per-unit hash rate (TH/s)
    asic_price: float = 5000,      # Per-unit cost (USD)
    asic_power_w: float = 3500,    # Per-unit power (Watts)
    electricity_cost: float = 0.07, # USD/kWh
    attack_duration_hours: float = 6  # Duration of the attack
) -> dict:
    """
    Calculate the cost to acquire 51% of the network hash rate.
    """
    network_hashrate_th = network_hashrate_ehs * 1e6  # Convert EH/s to TH/s
    
    # Hash rate needed for 51%
    # If network is N, attacker needs A such that A / (N + A) > 0.51
    # A > 0.51N / 0.49 ~= 1.0408N
    # So attacker needs slightly more than the entire current network!
    # More commonly quoted: attacker needs to ADD 51% of current to get majority
    # Assuming attacker starts from zero:
    attack_hashrate_th = network_hashrate_th * (0.51 / 0.49)
    
    # Number of ASICs needed
    n_asics = int(np.ceil(attack_hashrate_th / asic_hashrate_th))
    
    # Capital cost (hardware)
    hardware_cost = n_asics * asic_price
    
    # Electricity cost for attack duration
    total_power_kw = (n_asics * asic_power_w) / 1000
    electricity_total = total_power_kw * attack_duration_hours * electricity_cost
    
    # Total attack cost
    total_cost = hardware_cost + electricity_total
    
    return {
        'attack_hashrate_th': attack_hashrate_th,
        'attack_hashrate_ehs': attack_hashrate_th / 1e6,
        'n_asics': n_asics,
        'hardware_cost': hardware_cost,
        'electricity_cost': electricity_total,
        'total_cost': total_cost,
        'total_power_gw': total_power_kw / 1e6,
    }


# Current Bitcoin network
attack = attack_cost_analysis(500)  # 500 EH/s

print("=== 51% Attack Cost Analysis (Bitcoin @ 500 EH/s) ===")
print(f"\nAttacker hash rate needed: {attack['attack_hashrate_ehs']:.1f} EH/s")
print(f"ASICs required: {attack['n_asics']:,} units (Antminer S21)")
print(f"\nHardware cost: ${attack['hardware_cost']:,.0f} ({attack['hardware_cost']/1e9:.1f}B)")
print(f"Electricity cost (6h): ${attack['electricity_cost']:,.0f}")
print(f"Total attack cost: ${attack['total_cost']:,.0f} ({attack['total_cost']/1e9:.1f}B)")
print(f"\nTotal power required: {attack['total_power_gw']:.2f} GW")
print(f"(For reference, a large nuclear plant produces ~1 GW)")

In [ ]:
# === Attack Cost vs Potential Gain ===

# Double-spend scenario: attacker sends BTC to exchange, buys something,
# then reorgs to reverse the transaction
btc_price = 60000
confirmations = [1, 2, 3, 6, 12, 24]

# Probability of successful reorg decreases with confirmations
# and attacker's fraction of hash rate
def reorg_probability(attacker_fraction: float, n_confirmations: int) -> float:
    """
    Probability that attacker can build a longer chain.
    Simplified model based on random walk / Nakamoto's analysis.
    If q > 0.5: probability = 1 (guaranteed)
    If q < 0.5: probability = (q/p)^n where p = 1-q
    """
    q = attacker_fraction
    p = 1 - q
    if q >= 0.5:
        return 1.0
    return (q / p) ** n_confirmations


# Attack cost scales with network hash rate
hash_rates_ehs = np.linspace(50, 1000, 50)
costs = [attack_cost_analysis(hr)['total_cost'] for hr in hash_rates_ehs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Attack cost vs network hash rate
ax1.plot(hash_rates_ehs, np.array(costs) / 1e9, 'r-', linewidth=2.5)
ax1.set_xlabel('Network Hash Rate (EH/s)', fontsize=13)
ax1.set_ylabel('51% Attack Cost (Billion USD)', fontsize=13)
ax1.set_title('Attack Cost Scales Linearly with Hash Rate', fontsize=14)
ax1.axvline(x=500, color='blue', linestyle='--', alpha=0.5, label='Current ~500 EH/s')
ax1.legend(fontsize=11)

# Right: Reorg probability vs attacker hash fraction
fractions = np.linspace(0.01, 0.49, 100)
for n_conf in [1, 2, 3, 6]:
    probs = [reorg_probability(f, n_conf) for f in fractions]
    ax2.plot(fractions * 100, probs, linewidth=2, label=f'{n_conf} confirmations')

ax2.set_xlabel('Attacker Hash Rate (% of network)', fontsize=13)
ax2.set_ylabel('Reorg Probability', fontsize=13)
ax2.set_title('Double-Spend Success Probability', fontsize=14)
ax2.legend(fontsize=11)
ax2.set_yscale('log')
ax2.set_ylim(1e-10, 1)

plt.tight_layout()
plt.show()

# Expected value calculation
print("\n=== Attack Economics ===")
print(f"\nScenario: Double-spend a $10M exchange deposit")
print(f"Attack cost (hardware + 6h electricity): ${attack['total_cost']/1e9:.1f}B")
print(f"Potential gain: $10,000,000")
print(f"Cost-to-gain ratio: {attack['total_cost'] / 10e6:.0f}:1")
print(f"\nConclusion: A 51% attack on Bitcoin is economically irrational.")

In [ ]:
# === Comparative Attack Costs Across Hypothetical Network Sizes ===

networks = [
    ('Small PoW Chain (1 EH/s)', 1),
    ('Medium Chain (50 EH/s)', 50),
    ('Bitcoin Cash (~5 EH/s)', 5),
    ('Litecoin equiv. (~1 EH/s)', 1),
    ('Bitcoin (~500 EH/s)', 500),
]

print(f"{'Network':<35} {'Hash Rate':>12} {'Attack Cost':>16} {'ASICs Needed':>14}")
print('=' * 80)
for name, hr in sorted(networks, key=lambda x: x[1]):
    a = attack_cost_analysis(hr)
    cost_str = f"${a['total_cost']/1e6:,.1f}M" if a['total_cost'] < 1e9 else f"${a['total_cost']/1e9:,.1f}B"
    print(f"{name:<35} {hr:>10} EH/s {cost_str:>16} {a['n_asics']:>14,}")

print("\nSmaller PoW networks are dramatically more vulnerable to 51% attacks.")
print("This is why hash rate is a crucial security metric.")

**Key Insight**: The cost of a 51% attack on Bitcoin is measured in **tens of billions of dollars** in hardware alone, dwarfing any plausible gain from double-spending. This massive cost serves as Bitcoin's security budget. Smaller PoW chains, however, can be attacked for a fraction of the cost, which is why several have suffered real 51% attacks historically.

---
# Exercises

The following exercises reinforce the concepts from this notebook. Each includes starter code for you to complete.

## Exercise 1: Mining Farm ROI Over 2 Years

Model a mining farm with 100 Antminer S21 units. Assume:
- Initial investment: 100 x $5,000 = $500,000 (hardware) + $50,000 (infrastructure)
- Electricity: $0.06/kWh
- Difficulty increases 5% every 2 weeks
- BTC price stays at $60,000

Calculate monthly profits and cumulative ROI over 24 months. When does the farm break even?

In [ ]:
# === Exercise 1: Mining Farm ROI ===

def mining_farm_roi(
    n_units: int = 100,
    unit_cost: float = 5000,
    infrastructure_cost: float = 50000,
    hash_rate_per_unit: float = 200,  # TH/s
    power_per_unit: float = 3500,      # Watts
    electricity_cost: float = 0.06,    # $/kWh
    initial_difficulty: float = 75e12,
    difficulty_increase_pct: float = 5.0,  # % per 2-week period
    btc_price: float = 60000,
    block_reward: float = 3.125,
    months: int = 24
):
    """
    TODO: Calculate monthly revenue, costs, profit, and cumulative ROI.
    
    Hints:
    - There are ~2 difficulty adjustments per month
    - Use the mining_profitability() function from Part 1
    - Track cumulative profit vs initial investment
    """
    total_investment = n_units * unit_cost + infrastructure_cost
    total_hashrate = n_units * hash_rate_per_unit
    total_power = n_units * power_per_unit
    
    monthly_records = []
    cumulative_profit = -total_investment  # Start negative (initial investment)
    difficulty = initial_difficulty
    
    for month in range(1, months + 1):
        # YOUR CODE HERE:
        # 1. Calculate monthly profit using mining_profitability()
        # 2. Update cumulative profit
        # 3. Adjust difficulty (2 adjustments per month)
        # 4. Append to monthly_records
        pass
    
    # TODO: Plot monthly profit and cumulative ROI
    # TODO: Find break-even month
    
    return monthly_records


# Uncomment to test:
# results = mining_farm_roi()
print("Exercise 1: Implement the mining_farm_roi() function above.")

## Exercise 2: Transition from Subsidy to Fee-Based Security

As block rewards halve, transaction fees must eventually sustain the network's security budget. Model this transition:
- Project the block subsidy over the next 30 years
- Assume transaction fees grow linearly from 0.5 BTC/block to some target
- What average fee level (in BTC) is needed to maintain current miner revenue levels?

In [ ]:
# === Exercise 2: Fee-Based Security Model ===

def fee_security_model(years_forward: int = 30, current_fees_btc: float = 0.5):
    """
    TODO: Model the transition from subsidy to fee-based miner revenue.
    
    Steps:
    1. Calculate block subsidy at each year from now to now + years_forward
    2. Calculate the fee level needed to maintain current total miner revenue
       (current_subsidy + current_fees) * blocks_per_year * btc_price
    3. Plot subsidy, required fees, and total revenue over time
    """
    current_block_height = 840_000  # Approximate (post-2024 halving)
    blocks_per_year = 365.25 * 144
    
    # Current total revenue per block
    current_subsidy = block_reward(current_block_height)
    current_revenue_per_block = current_subsidy + current_fees_btc
    
    # YOUR CODE HERE:
    # 1. For each year, compute the block height and subsidy
    # 2. Calculate the fee needed: required_fees = current_revenue_per_block - future_subsidy
    # 3. Plot the results
    pass


# Uncomment to test:
# fee_security_model()
print("Exercise 2: Implement the fee_security_model() function above.")

## Exercise 3: Hash Rate Needed to Find One Block Per Week

Given the current difficulty, calculate the hash rate a solo miner would need to expect finding one block per week on average. Express the answer in TH/s and in number of Antminer S21 units.

In [ ]:
# === Exercise 3: Solo Block Discovery ===

def hashrate_for_weekly_block(difficulty: float = 75e12):
    """
    TODO: Calculate the hash rate needed to find one block per week on average.
    
    Use the formula: E[blocks/week] = (H * seconds_per_week) / (D * 2^32)
    Set E[blocks/week] = 1 and solve for H.
    """
    seconds_per_week = 7 * 24 * 3600
    
    # YOUR CODE HERE:
    # 1. Solve for H
    # 2. Convert to TH/s
    # 3. Calculate number of Antminer S21 units
    # 4. Print results
    pass


# Uncomment to test:
# hashrate_for_weekly_block()
print("Exercise 3: Implement the hashrate_for_weekly_block() function above.")

## Exercise 4: Compare Profitability Across ASIC Models

Compare three different ASIC models and determine which is most profitable at different electricity prices. Create a visualization showing the crossover points.

In [ ]:
# === Exercise 4: ASIC Comparison ===

asic_models = {
    'Antminer S21': {'hash_rate_th': 200, 'power_watts': 3500, 'price_usd': 5000},
    'Antminer S19 XP': {'hash_rate_th': 140, 'power_watts': 3010, 'price_usd': 3500},
    'Whatsminer M50S': {'hash_rate_th': 126, 'power_watts': 3276, 'price_usd': 3000},
}


def compare_asics(models: dict, difficulty: float = 75e12,
                   btc_price: float = 60000, block_reward: float = 3.125):
    """
    TODO: Compare profitability of different ASIC models.
    
    Steps:
    1. For each model, calculate daily profit across electricity costs $0.02-$0.15/kWh
    2. Plot daily profit vs electricity cost for each model
    3. Find and annotate crossover points
    4. Calculate efficiency (J/TH) for each model
    5. Determine which model is best at different electricity prices
    """
    # YOUR CODE HERE
    pass


# Uncomment to test:
# compare_asics(asic_models)
print("Exercise 4: Implement the compare_asics() function above.")

# Print efficiency for reference
print("\nASIC Efficiency Reference:")
for name, specs in asic_models.items():
    eff = specs['power_watts'] / specs['hash_rate_th']
    print(f"  {name}: {eff:.1f} J/TH ({specs['hash_rate_th']} TH/s, {specs['power_watts']}W)")

---
# Summary

In this notebook, we explored five dimensions of mining economics:

1. **Profitability Calculator**: Mining is a thin-margin business dominated by electricity costs and BTC price. Geographic arbitrage on electricity is the primary competitive advantage.

2. **Difficulty Adjustment**: Bitcoin's self-regulating difficulty mechanism maintains ~10-minute block times despite hash rate fluctuations, though with a significant lag during sudden drops.

3. **Supply Schedule**: Bitcoin's programmatic monetary policy creates a predictable disinflationary supply curve, with inflation dropping below gold's after the third halving.

4. **Pool Economics**: Mining pools reduce variance dramatically. PPS offers the most predictable income (pool absorbs risk), while PPLNS shares risk between pool and miners.

5. **51% Attack Costs**: Attacking Bitcoin costs tens of billions of dollars, making it economically irrational. Smaller PoW chains are far more vulnerable.

## Next Steps

- Continue to **sections/02-bitcoin-deep-dive.md** for a comprehensive look at Bitcoin's protocol, transaction structure, and scripting system.
- Explore how mining economics interact with Bitcoin's fee market and the long-term security model as block subsidies diminish.
- Consider the environmental implications of Proof-of-Work and compare to Proof-of-Stake security models.